# WSP Toolkit — M3 demo on Kaggle: Qwen3.5-0.8B base vs. the WSP edit (v2)

Compares the base model with the authors' edited model (the "loop fix") under the paper's sampling settings, then
runs non-edit baselines with whatever time is left. Results go to `/kaggle/working/` (download from the **Output** tab).

**Before running:** right-hand panel → **Accelerator: GPU T4 x2**, **Internet: On** (phone-verified account).
Then **Save Version → Save & Run All (Commit)** so it runs in the background.

**How time is managed (v2):** with batching, a run takes as long as its longest answer, so the cost of a run is
`ceil(LIMIT / BATCH) × MAX_NEW_TOKENS ÷ decode-steps-per-second`, not the number of problems. The notebook
measures decode speed after a warm-up, starts runs in priority order (base, edited, then baselines) on whichever GPU
is free, only starts a run that can finish before `DEADLINE_HOURS`, and stops anything still running at the deadline.
The report is always written from the runs that completed.

In [ ]:
# ---- settings ---------------------------------------------------------------
BATCH = 8                   # problems generated together per GPU (8 fits a T4 in float32 at 32k tokens)
LIMIT = 8                   # MATH-500 problems per run; keep it a multiple of BATCH (extra problems in a batch are free)
MAX_NEW_TOKENS = 32768      # paper setting; the base model hits this on ~93% of problems in the paper
SUITES = "math500_heldout"
DTYPE = "float32"           # T4 has no bfloat16; float16 may overflow in Qwen
DEADLINE_HOURS = 11.0       # hard stop for generation (Kaggle kills the session at 12 h and outputs may be lost)
SAFETY = 1.3                # long contexts decode slower than the short warm-up measurement
RUNS = ["base", "edited", "baseline:repetition_penalty_1.1", "baseline:presence_penalty_2.0",
        "baseline:temperature_1.0"]   # priority order
REPO = "https://github.com/encore488/weight-space-manipulator.git"
BRANCH = "main"
import time; T0 = time.time()


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; N_GPU = torch.cuda.device_count(); print("GPUs:", N_GPU)
assert N_GPU >= 1, "No GPU: set Accelerator to 'GPU T4 x2' in the notebook settings"


In [ ]:
%cd /kaggle/working
!rm -rf weight-space-manipulator && git clone -q -b $BRANCH $REPO
%cd /kaggle/working/weight-space-manipulator
!pip install -q -U "transformers>=5" huggingface_hub pyarrow 2>&1 | grep -v "requires\|incompatible" | tail -2
!pip install -q --no-deps -e . 2>&1 | tail -2
!pip install -q flash-linear-attention 2>&1 | tail -1 || echo "flash-linear-attention not installed (slower)"
# optional faster conv kernel; uses a prebuilt wheel when one matches, gives up after 10 minutes
!timeout 600 pip install -q causal-conv1d --no-build-isolation 2>&1 | tail -1 || echo "causal-conv1d not installed (slower)"
import transformers; print("transformers", transformers.__version__)


In [ ]:
from huggingface_hub import snapshot_download
pat = ["*.json", "*.safetensors", "*.txt", "*.jinja", "tokenizer*"]
snapshot_download("Qwen/Qwen3.5-0.8B", revision="2fc06364715b967f1860aea9cf38778875588b17", allow_patterns=pat)
snapshot_download("RikiyaT/Qwen3.5-0.8B-weight-space-programming", allow_patterns=pat)
print("downloaded")


In [ ]:
# measure decode speed (after a warm-up) on GPU 0
import os, json, subprocess, math, glob
W = "/kaggle/working"
env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0", "WSP_DATA_DIR": f"{W}/wsp_smoke"}
subprocess.run(["python", "scripts/qwen_m3_demo.py", "--confirm-download", "--device", "cuda", "--dtype", DTYPE,
                "--batch-size", str(BATCH), "--smoke", "--runs", "base", "--suites", SUITES.split(",")[0],
                "--results-jsonl", f"{W}/smoke.jsonl"], env=env, check=True)
steps_per_s = json.loads(open(f"{W}/smoke.jsonl").read().splitlines()[-1])["decode_steps_per_second"]
RUN_HOURS = math.ceil(LIMIT / BATCH) * MAX_NEW_TOKENS * len(SUITES.split(",")) / steps_per_s * SAFETY / 3600 + 0.05
left = DEADLINE_HOURS - (time.time() - T0) / 3600
print(f"decode speed {steps_per_s} steps/s -> worst case {RUN_HOURS:.2f} h per run; {left:.1f} h left; "
      f"about {int(left // RUN_HOURS) * N_GPU} runs fit")
if RUN_HOURS > left:
    print("WARNING: even one run may not finish. Lower MAX_NEW_TOKENS (e.g. 16384) and loop/limit rates will still "
          "be measured at that cap.")


In [ ]:
# scheduler: one run per process; start the next run in priority order on a free GPU if it can finish in time
queue = list(RUNS)
running = {}   # gpu -> (run, Popen, start_time)
done, skipped, stopped = [], [], []
common = ["python", "scripts/qwen_m3_demo.py", "--confirm-download", "--device", "cuda", "--dtype", DTYPE,
          "--batch-size", str(BATCH), "--limit", str(LIMIT), "--max-new-tokens", str(MAX_NEW_TOKENS),
          "--suites", SUITES]
last_print = 0
while queue or running:
    now_h = (time.time() - T0) / 3600
    for gpu in [str(g) for g in range(N_GPU)]:
        if gpu in running and running[gpu][1].poll() is not None:
            run, p, st = running.pop(gpu)
            (done if p.returncode == 0 else stopped).append(run)
            print(f"[{now_h:.2f} h] GPU {gpu} finished {run} in {(time.time() - st) / 3600:.2f} h (exit {p.returncode})")
        if gpu not in running and queue:
            if now_h + RUN_HOURS > DEADLINE_HOURS:
                skipped += queue; queue = []
                print(f"[{now_h:.2f} h] not enough time left for {skipped}; skipping them")
                continue
            run = queue.pop(0)
            env = {**os.environ, "CUDA_VISIBLE_DEVICES": gpu, "WSP_DATA_DIR": f"{W}/wsp_gpu{gpu}"}
            log = open(f"{W}/log_{run.replace(':', '_')}.txt", "w")
            running[gpu] = (run, subprocess.Popen(common + ["--runs", run, "--results-jsonl",
                            f"{W}/results_{run.replace(':', '_')}.jsonl"], env=env, stdout=log,
                            stderr=subprocess.STDOUT), time.time())
            print(f"[{now_h:.2f} h] GPU {gpu} started {run}")
    if now_h > DEADLINE_HOURS:
        for gpu, (run, p, st) in list(running.items()):
            p.kill(); stopped.append(run)
            print(f"[{now_h:.2f} h] deadline: stopped {run} on GPU {gpu} (incomplete, not reported)")
        running = {}
        break
    if time.time() - last_print > 1800:
        last_print = time.time()
        print(f"[{now_h:.2f} h] running: {[r for r, _, _ in running.values()]}, done: {done}, queued: {queue}")
    time.sleep(60)
print("done:", done, "| stopped:", stopped, "| skipped:", skipped)


In [ ]:
# report from completed runs (+ weight diff)
files = sorted(glob.glob(f"{W}/results_*.jsonl"))
print(files)
if files:
    subprocess.run(["python", "scripts/qwen_m3_demo.py", "--report-from", *files, "--with-diff",
                    "--out", f"{W}/m3_qwen_demo.md"], env={**os.environ, "WSP_DATA_DIR": f"{W}/wsp_gpu0",
                                                          "CUDA_VISIBLE_DEVICES": ""}, check=True)


## Send the results back

Download `m3_qwen_demo.md`, all `results_*.jsonl` and `log_*.txt` from the **Output** tab and upload them to the
Claude Code session.